# 05 — SHAP Explainability

Generate global and local SHAP explanations for the best model.

**Requirements:** FR-15 through FR-18 (PRD)

In [ ]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('..').resolve()))
from src.explainability import plot_waterfall, run_shap_analysis
from src.feature_engineering import engineer_features, get_feature_columns
from src.modeling import train_test_split_data
from src.utils import data_path, results_path, set_seed

set_seed()
FIG_DIR = results_path('figures')
MODEL_DIR = results_path('models')

In [ ]:
# Load data and trained model
df = pd.read_csv(data_path('processed', 'oasis_merged_final.csv'))
if 'BrainAtrophyRatio' not in df.columns:
    df = engineer_features(df)

feature_cols = [c for c in get_feature_columns() if c in df.columns]
X = df[feature_cols]
y = df['target']
_, X_test, _, y_test = train_test_split_data(X, y)

pipeline = joblib.load(MODEL_DIR / 'best_model.pkl')
model = pipeline.named_steps['model']

In [ ]:
# FR-15, FR-16: Global SHAP bar + beeswarm plots
shap_results = run_shap_analysis(model, X_test, output_dir=FIG_DIR)
print('Top features:', shap_results['top_features'])

In [ ]:
# FR-17: Waterfall plots for one true positive and one false negative
y_pred = pipeline.predict(X_test)
tp_idx = np.where((y_test.values == 1) & (y_pred == 1))[0]
fn_idx = np.where((y_test.values == 1) & (y_pred == 0))[0]

if len(tp_idx) > 0:
    plot_waterfall(shap_results['explainer'], shap_results['shap_values'], X_test, tp_idx[0],
                   save_path=FIG_DIR / 'shap_waterfall_true_positive.png')
if len(fn_idx) > 0:
    plot_waterfall(shap_results['explainer'], shap_results['shap_values'], X_test, fn_idx[0],
                   save_path=FIG_DIR / 'shap_waterfall_false_negative.png')

## Clinical Interpretation (FR-18)

For each of the top 5 SHAP features, document:
1. Biological meaning
2. Direction of influence on dementia prediction
3. Alignment with published medical evidence

See `docs/Biology_Topics_Study_Guide.md` for background.